In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install -U langchain langchain-groq groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.2/111.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 496.3/496.3 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.0/342.0 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.6/212.6 kB 18.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.27
    Uninstalling langchain-0.3.27:
      Successfully uninstalled langchain-0.3.27


In [2]:
import os
os.environ["GROQ_API_KEY"] = "your_api_key"

In [6]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Chain

In [11]:
prompt = PromptTemplate(
    template='What is the capital and currency of the country {country}',
    input_variables=['country']
)

model = ChatGroq(model="groq/compound")
parser = StrOutputParser()
chain = prompt | model | parser

In [13]:
result = chain.invoke({'country':'India'})
print(result)

**Answer**

- **Capital of India:** **New Delhi**  
- **Currency of India:** **Indian Rupee (INR)**  

---

### How I arrived at this answer  

1. **Performed a web search** for “What is the capital and currency of India?”  
2. **Key findings from the search results**:  

   - **SimCorner travel guide** lists New Delhi as the capital and the Indian Rupee as the currency.  
   - **Wikipedia entry for “Indian rupee”** confirms that the Indian Rupee (symbol ₹, ISO code INR) is the official currency of India.  
   - **Reserve Bank of India FAQ** also states that the Indian currency is called the Indian Rupee (INR).  

3. **Cross‑checking** the information from multiple reputable sources (travel guide, Wikipedia, RBI) shows consistent data:  

   - **Capital:** New Delhi (the seat of the President, Parliament, and central government).  
   - **Currency:** Indian Rupee (₹), subdivided into 100 paise, issued and regulated by the Reserve Bank of India.  

Since all sources agree, the definitiv

# Sequential Chain

In [19]:
prompt1 = PromptTemplate(
    template='Generate a 7 line description on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate 3 questions form the following text \n {text}',
    input_variables=['text']
)

In [20]:
chain = prompt1 | model | parser | prompt2 | model |parser

result = chain.invoke({'topic':'AI in medical field'})
print(result)

Below is the complete response that includes the full reasoning you asked for, the concise 7‑line description of AI in the medical field, and the three questions generated from that material.

---

## 1. Reasoning and sources used  

**Research step** – I performed a web‑search for “AI in medical field” and gathered several high‑quality sources (ForeSeeMed, Spectral‑AI, IBM, Wikipedia, FDA, EU health portal, etc.).  

**Key themes extracted**  

| Theme | What the sources say |
|-------|----------------------|
| **Diagnostic power** | AI can read radiology images, detect cancers, heart disease, pneumonia, skin lesions, etc., often matching or surpassing human experts. |
| **Personalized treatment & clinical trials** | Machine‑learning models predict outcomes, help design trials, and tailor therapies to individual patients. |
| **Administrative efficiency** | AI automates data entry, claims processing, scheduling, and transcription, freeing clinicians for direct patient care. |
| **Drug

# Parallel Chain

In [23]:
from langchain_core.runnables import RunnableParallel

In [26]:
sentiment_prompt = PromptTemplate(
    template='Sentiment (positive/negative/neutral) : {text}',
    input_variables=['Text']
)

summary_prompt = PromptTemplate(
    template='One line summary : {text}',
    input_variables=['text']
)

In [27]:
sentiment_chain = sentiment_prompt | model | parser
summary_chain = summary_prompt | model | parser

### neutral

In [28]:
parallel_chain = RunnableParallel(
    sentiment = sentiment_chain,
    summary = summary_chain
)

result = parallel_chain.invoke({
    'text':'The phone is very good but it is little expensive.'
})

print(result)

{'sentiment': '**Analysis of the statement**\n\nThe sentence *“The phone is very good but it is little expensive.”* contains two contrasting clauses:\n\n1. **Positive part:** “The phone is very good.” – This is a clear positive sentiment.\n2. **Negative part:** “but it is little expensive.” – This introduces a negative sentiment, though the word *“little”* softens the negativity compared with stronger terms like “very expensive”.\n\n**Balancing the sentiments**\n\n- The positive clause uses a strong qualifier (*very good*), indicating a strong positive feeling.\n- The negative clause is milder (*little expensive*), suggesting a mild complaint about price.\n\nBecause the statement includes both a positive and a negative component, many sentiment‑analysis systems would look for the dominant sentiment. In this case the positive sentiment is stronger than the negative one, but the presence of a clear drawback prevents the overall sentiment from being unequivocally positive.\n\n**Conclusion

### positive

In [29]:
result = parallel_chain.invoke({
    'text':'Excellent phone, Display superb.Camera good.Performance fast.Overall satisfied with upgrade to S23 Ultra.'
})

print(result)

{'sentiment': '**Sentiment:** Positive.', 'summary': 'Excellent phone with a superb display, good camera, fast performance—very satisfied with the S23\u202fUltra upgrade.'}


# Conditional Chain

In [39]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

### sentiment pormpt and chain

In [46]:
sentiment_prompt = PromptTemplate(
    template='Sentiment (positive/negative/neutral) : {text}',
    input_variables=['text']
)

classifier_chain = sentiment_prompt | model | parser

### postive prompt and chain

In [58]:
pos_prompt = PromptTemplate(
    template='Thanks for your feedback! Please visit again!',
    input_variables = ['text']
)

pos_chain = pos_prompt | model | parser

### negative prompt and chain

In [53]:
neg_prompt = PromptTemplate(
    template="Sorry for the issue. We will try to solve it",
    input_variables = ['text']
)

neg_chain = neg_prompt | model | parser

### conditional prompt and chain

In [63]:
conditional_chain = RunnableBranch(
    (lambda x : 'positive' in x["sentiment"].lower(), pos_chain),
    (lambda x : 'negative' in x["sentiment"].lower(), neg_chain),
    RunnableLambda(lambda x : 'Sorry could not find sentiment')  #default msg
)

### complete chain

In [61]:
def review_chain(review_text):
    sentiment = sentiment_chain.invoke({"text": review_text})
    
    result = conditional_chain.invoke({
        "sentiment": sentiment,
        "review": review_text
    })
    
    return {
        "sentiment": sentiment,
        "response": result
    }

### positive sentiment

In [64]:
result = review_chain("The app is very smooth and useful")
print(result)

{'sentiment': 'Positive', 'response': 'You’re welcome! Feel free to stop by again anytime. Have a great day!'}


### negative sentiment

In [65]:
result = review_chain("the shirt is of very bad quality. the colth shrinked after one wash")
print(result)

{'sentiment': 'Negative', 'response': 'No worries at all—just let me know what you need, and I’ll do my best to help!'}


### neutral sentiment

In [66]:
result = review_chain("Delhi is the capital of India")
print(result)

{'sentiment': 'The sentiment of the statement “Delhi is the capital of India” is **neutral**.', 'response': 'Sorry could not find sentiment'}
